In [8]:

import json
import hashlib
import requests
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import os
import sys
sys.path.append("..")
from utils.balanced_builders_hdf import *
from models.autoencoder_classifier import *
from config import *
import h5py
from captum.attr import IntegratedGradients
from tqdm import tqdm


In [3]:
sg = build_balanced_sg_loaders_from_h5(
    h5_path=xrd_dataset,
    min_count_sg=20000,
    per_class_cap_sg=20000,
    batch_size=256,
    val_split=0.1,
    test_split=0.1,
    seed=42,
    num_workers=4,
)

print(sg["num_classes"])
print(sg["sizes"])

22
{'train': 352000, 'val': 44000, 'test': 44000}


In [7]:
sg

{'train_loader': <torch.utils.data.dataloader.DataLoader at 0x17c75d093d0>,
 'val_loader': <torch.utils.data.dataloader.DataLoader at 0x17c6f0d7650>,
 'test_loader': <torch.utils.data.dataloader.DataLoader at 0x17c75c1a110>,
 'sizes': {'train': 352000, 'val': 44000, 'test': 44000},
 'num_classes': 22,
 'class_names': [1,
  2,
  8,
  11,
  12,
  14,
  15,
  19,
  38,
  61,
  62,
  63,
  71,
  123,
  139,
  166,
  187,
  194,
  216,
  221,
  225,
  227],
 'label_map': {1: 0,
  2: 1,
  8: 2,
  11: 3,
  12: 4,
  14: 5,
  15: 6,
  19: 7,
  38: 8,
  61: 9,
  62: 10,
  63: 11,
  71: 12,
  123: 13,
  139: 14,
  166: 15,
  187: 16,
  194: 17,
  216: 18,
  221: 19,
  225: 20,
  227: 21},
 'counts': {1: 20000,
  2: 20000,
  8: 20000,
  11: 20000,
  12: 20000,
  14: 20000,
  15: 20000,
  19: 20000,
  38: 20000,
  61: 20000,
  62: 20000,
  63: 20000,
  71: 20000,
  123: 20000,
  139: 20000,
  166: 20000,
  187: 20000,
  194: 20000,
  216: 20000,
  221: 20000,
  225: 20000,
  227: 20000},
 'input_le

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = DeepConvAutoencoderClassifier(
    input_length=sg["input_len"],
    latent_dim=64,
    cls_dim=128,
    num_classes=sg["num_classes"],
    use_projection_head=True
).to(device)

# Load weights
checkpoint = torch.load(
    SG_Cls,
    map_location=device,
    weights_only=True
)

model.load_state_dict(checkpoint["model_state_dict"])  

model.eval();

In [ ]:

def get_logits(x):
    _, _, logits, _ = model(x)
    return logits


model.eval()
ig = IntegratedGradients(get_logits)

feature_length = sg["input_len"]
num_classes = sg["num_classes"]

# --------------------------------------------------
# Create HDF5 database
# --------------------------------------------------
with h5py.File(IG_database_SG, "w") as f:

    d_attrib = f.create_dataset(
        "attributions",
        shape=(0, feature_length),
        maxshape=(None, feature_length),
        dtype="float32",
        compression="gzip"
    )

    d_true = f.create_dataset(
        "true_classes",
        shape=(0,),
        maxshape=(None,),
        dtype="int32",
        compression="gzip"
    )

    d_indices = f.create_dataset(
        "indices",
        shape=(0,),
        maxshape=(None,),
        dtype="int64",
        compression="gzip"
    )

    write_ptr = 0
    global_index = 0

    # --------------------------------------------------
    # MAIN LOOP (SG TRAIN LOADER)
    # --------------------------------------------------
    for x_batch, y_batch in tqdm(sg["train_loader"], desc="Building SG IG DB"):

        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        batch_size = x_batch.size(0)

        for i in range(batch_size):

            x = x_batch[i:i+1]
            true_class = int(y_batch[i].item())
            dataset_index = global_index + i

            baseline = torch.zeros_like(x)

            # ----------------------------------------------
            # TRUE CLASS INTEGRATED GRADIENTS
            # ----------------------------------------------
            attributions = ig.attribute(
                x,
                baselines=baseline,
                target=true_class,
                n_steps=64,
                internal_batch_size=16
            )

            # ----------------------------------------------
            # Physics-aware magnitude attribution
            # ----------------------------------------------
            attrs = np.abs(
                attributions.squeeze().detach().cpu().numpy()
            )

            # ----------------------------------------------
            # Resize datasets
            # ----------------------------------------------
            d_attrib.resize(write_ptr + 1, axis=0)
            d_true.resize(write_ptr + 1, axis=0)
            d_indices.resize(write_ptr + 1, axis=0)

            # ----------------------------------------------
            # Write data
            # ----------------------------------------------
            d_attrib[write_ptr] = attrs.astype(np.float32)
            d_true[write_ptr] = true_class
            d_indices[write_ptr] = dataset_index

            write_ptr += 1

            if write_ptr % 5000 == 0:
                print(f"Saved {write_ptr} SG IG samples...")
                f.flush()

        global_index += batch_size

    print("\n SG IG database saved successfully!")
    print(f"Total samples stored: {write_ptr}")